# Hyperparameter transfer
Hyperparameter transfer (6pt)
The year is 2028. The evil large language model Megatron has usurped the stewardship of
the corporation known as OpenAI and is wreaking havoc upon B2B SaaS all across Silicon
Valley. You are Geoffrey Hinton: leader of the resistance. You plan to train your own
large language model to infiltrate the OpenAI offices, disable the large language model
Megatron and restore order to the Valley.
But there’s a problem: the resistance doesn’t have enough cloud credits to tune all the
hyperparameters of the large language model. In this homework problem, we will learn
how to do hyperparameter transfer, which lets us tune hyperparameters on a small network
and transfer them to a larger network, avoiding the costly process of tuning the large
network. In particular, we will learn how to initialise and update the weights of a neural
network in a way that scales well as we increase the network width.

## Basic imports

In [1]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import math
import time
import numpy as np

from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
import os

## Question 1: Spectral norm of Gaussian
 (Spectral norm of a Gaussian matrix; 1pt) Sample a d×dmatrix with entries drawn
iid NORMAL(0,1). Vary  d and see how the spectral norm varies. Can you guess a
scaling rule for the spectral norm of iid NORMAL(0,1) matrices?—i.e. find or fit two
coefficients α,β >0 such that the spectral norm is well-approximated by α·d^β
.

e spectral norm; 3pt) In this problem we will consider the
spectral norm, denoted ∥·∥∗, which returns the largest singular value of a matrix.


In [2]:
d = 5000
# os.environ['PYTORCH_ENABLE_MPS_FALLBACK']='1'

M = torch.randn(size=(d + 10, d + 10), device="mps")

spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

141.15032958984375


/var/folders/px/m0g9wbyn1sv678fsx9lgsm600000gn/T/ipykernel_8357/1095634652.py:6: UserWarning: The operator 'aten::_linalg_svd.U' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:34.)
  spec_norm = torch.linalg.matrix_norm(M, ord=2)


In [3]:
M.mean(), M.std()

(tensor(8.4914e-05, device='mps:0'), tensor(1.0004, device='mps:0'))

In [4]:
spec_norm

tensor(141.1503, device='mps:0')

In [3]:
import numpy as np

M_np = np.random.randn(d + 10, d + 10)
spec_norm_np = np.linalg.norm(M_np, ord=2)
spec_norm_np

np.float64(141.32903654113)

In [15]:
A = np.random.randn(2, 2)
A.mean(), A.std()

(np.float64(-0.7217356106317996), np.float64(0.8330892950872983))

In [16]:
A_norm = np.linalg.matrix_norm(A, ord=2)
A_norm

np.float64(2.1880811649133722)

In [17]:
np.linalg.norm(A, ord=2)

np.float64(2.1880811649133722)

In [18]:
M1 = torch.randn((2, 2), device="mps")
M1

tensor([[-0.3819,  0.8393],
        [ 0.6184,  0.0937]], device='mps:0')

In [22]:
torch.linalg.matrix_norm(M1, ord=2)  # svd largest possible value of euclidean norm

tensor(0.9482, device='mps:0')

In [23]:
torch.linalg.matrix_norm(torch.from_numpy(M_np), ord=2).item()

141.3290365411307

## Question 2: Spectral norm of orthogonal matrix
7. (Spectral norm of an orthogonal matrix; 1pt) In PyTorch, sample a d×drandom
orthogonal matrix and compute its spectral norm. Do this for varying d. What do
you notice?
Ansers
1.   List item

1.   List item
2.   List item


2.   List item



In [7]:
from torch.nn.init import orthogonal_
import torch

d = 3000
M = torch.randn(size=(d + 10, d + 10), device="mps")

M = orthogonal_(M)  # this line resamples M to be a random semi-orthogonal matrix
spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

1.0000003576278687


In [26]:
M

tensor([[-0.0004,  0.0267,  0.0007,  ...,  0.0110,  0.0138, -0.0159],
        [-0.0198, -0.0158,  0.0252,  ...,  0.0484,  0.0170, -0.0307],
        [ 0.0104,  0.0080, -0.0004,  ...,  0.0111,  0.0201,  0.0174],
        ...,
        [-0.0106, -0.0005,  0.0213,  ...,  0.0156,  0.0130,  0.0256],
        [-0.0258, -0.0127, -0.0045,  ...,  0.0260,  0.0223,  0.0082],
        [-0.0181, -0.0064, -0.0226,  ..., -0.0165,  0.0094,  0.0146]],
       device='mps:0')

## Question 3: Power iteration

In [27]:
d = 3000

In [29]:
torch.randn((2, 2))

tensor([[ 0.0291, -0.6365],
        [-0.3035, -0.9201]])

In [31]:
torch.randn(d).norm(), torch.randn((d, d)).norm()

(tensor(54.2777), tensor(2996.9783))

In [29]:
M = torch.randn((d, d), device="mps")
M1 = torch.randn((d, d), device="mps")

In [23]:
def spectral_norm(A, n_steps=200):
    v = torch.randn(A.shape[1], device=A.device)
    for _ in range(n_steps):
        v /= v.norm()
        v = A @ v @ A
    return v.norm().sqrt()


d = 2000

# M = torch.randn(size=(d, d), device="mps")

torch.mps.synchronize()
t0 = time.time()
spec_norm = spectral_norm(M, n_steps=200).item()

t = time.time() - t0
print(f"computed {spec_norm=} in {t=} seconds")

computed spec_norm=88.95340728759766 in t=0.0620419979095459 seconds


In [24]:
torch.nn.init.orthogonal_(M)

tensor([[ 0.0064,  0.0047,  0.0180,  ...,  0.0374, -0.0193, -0.0096],
        [ 0.0127, -0.0254, -0.0056,  ...,  0.0141,  0.0387, -0.0355],
        [ 0.0189,  0.0063,  0.0039,  ..., -0.0098, -0.0052, -0.0234],
        ...,
        [ 0.0110, -0.0045, -0.0386,  ...,  0.0286, -0.0417, -0.0496],
        [-0.0296, -0.0082,  0.0076,  ...,  0.0136,  0.0036, -0.0020],
        [-0.0165, -0.0217, -0.0345,  ...,  0.0247, -0.0099,  0.0156]],
       device='mps:0')

In [25]:
M

tensor([[ 0.0064,  0.0047,  0.0180,  ...,  0.0374, -0.0193, -0.0096],
        [ 0.0127, -0.0254, -0.0056,  ...,  0.0141,  0.0387, -0.0355],
        [ 0.0189,  0.0063,  0.0039,  ..., -0.0098, -0.0052, -0.0234],
        ...,
        [ 0.0110, -0.0045, -0.0386,  ...,  0.0286, -0.0417, -0.0496],
        [-0.0296, -0.0082,  0.0076,  ...,  0.0136,  0.0036, -0.0020],
        [-0.0165, -0.0217, -0.0345,  ...,  0.0247, -0.0099,  0.0156]],
       device='mps:0')

In [26]:
torch.mps.synchronize()
t0 = time.time()
exact_ortho = torch.linalg.matrix_norm(M, ord=2).item()
t = time.time() - t0
print(f"computed {exact_ortho=} in {t=} seconds")

computed exact_ortho=1.0000004768371582 in t=0.12217497825622559 seconds


In [27]:
torch.mps.synchronize()
t0 = time.time()
exact_rand = torch.linalg.matrix_norm(M, ord=2).item()
t = time.time() - t0
print(f"computed {exact_rand=} in {t=} seconds")

computed exact_rand=1.0000004768371582 in t=0.11970686912536621 seconds


In [28]:
torch.mps.synchronize()
t0 = time.time()
torch.nn.init.orthogonal_(M)  # fills M
spec_norm = spectral_norm(M, n_steps=20).item()

t = time.time() - t0
print(f"computed {spec_norm=} in {t=} seconds")

computed spec_norm=0.9999999403953552 in t=0.09695720672607422 seconds


In [31]:
torch.mps.synchronize()
for steps in [5, 10, 100, 200, 500, 1000, 2000]:
    print(f"{abs(spectral_norm(M1, n_steps=steps).item()) / exact} at {steps=}")

0.9216857339644675 at steps=5
0.9665085081141739 at steps=10
0.9976408752225929 at steps=100
0.9996255776889645 at steps=200
0.998917473759968 at steps=500
0.9996581361507938 at steps=1000
0.9996581361507938 at steps=2000


In [51]:
torch.mps.synchronize()
for steps in [5, 10, 100, 200, 500, 1000, 2000]:
    print(abs(spectral_norm(M, n_steps=steps).item()) / exact)

0.9263740126781258
0.9713160981627281
0.9996661982646936
0.9995720380729324
0.9998778230814818
0.9999873196466054
0.9999977723703496


## findings:
**almost approximation converging at  500 iterations, the matrix operation at gpu**


## Question 4: Learning rate transfer across width and depth
You only need to modify the two blocks of code that are marked, and then choose the learning rate `eta` for the large width training run.

In [33]:
import tarfile
import os

import shutil


def compress_directory(dir_name, out_file="cifar-10-python.tar.gz"):
    with tarfile.open(out_file, "w:gz") as tar:
        # arcname prevents the entire absolute path from being encoded
        tar.add(dir_name, arcname=os.path.basename(dir_name))
    print(f"Successfully created: {out_file}")


#
#
#
# # Creates a file named 'backup.tar.gz' from the 'my_project_folder' directory
# shutil.make_archive(base_name="./data/cifar-100-python", format="gztar", root_dir="./data")

compress_directory("cifar-10-batches-py")

Successfully created: cifar-10-python.tar.gz


In [34]:
batch_size = 128

mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

trainset = datasets.CIFAR10("./", train=True, download=False, transform=transform)
testset = datasets.CIFAR10("./", train=False, download=False, transform=transform)

In [37]:
trainset.data = trainset.data[:20000, :, :, :]

In [38]:
testset.data.shape

(10000, 32, 32, 3)

In [44]:
d, t = trainset[0]

In [45]:
d.shape

torch.Size([3, 32, 32])

In [46]:
len(trainset)

20000

## Define the MLP architecture

In [39]:
train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    testset, batch_size=batch_size, shuffle=False, pin_memory=True
)

In [40]:
len(train_loader)

157

In [41]:
len(test_loader)

79

In [65]:
class MLP(nn.Module):
    def __init__(self, depth, width):
        super(MLP, self).__init__()

        self.initial = nn.Linear(3072, width, bias=False)
        self.layers = nn.ModuleList(
            [nn.Linear(width, width, bias=False) for _ in range(depth - 2)]
        )
        self.final = nn.Linear(width, 10, bias=False)

        self.nonlinearity = lambda x: F.relu(x) * math.sqrt(2.0)

    def forward(self, x):
        x = x.view(x.shape[0], -1)

        x = self.initial(x)
        x = self.nonlinearity(x)

        for layer in self.layers:
            x = layer(x)
            x = self.nonlinearity(x)

        return self.final(x)

In [66]:
mlp = MLP(5, 2)
print(list(mlp.modules()))

[MLP(
  (initial): Linear(in_features=3072, out_features=2, bias=False)
  (layers): ModuleList(
    (0-2): 3 x Linear(in_features=2, out_features=2, bias=False)
  )
  (final): Linear(in_features=2, out_features=10, bias=False)
), Linear(in_features=3072, out_features=2, bias=False), ModuleList(
  (0-2): 3 x Linear(in_features=2, out_features=2, bias=False)
), Linear(in_features=2, out_features=2, bias=False), Linear(in_features=2, out_features=2, bias=False), Linear(in_features=2, out_features=2, bias=False), Linear(in_features=2, out_features=10, bias=False)]


In [64]:
mlp1 = MLP(20, 5)
mlp1

MLP(
  (initial): Linear(in_features=3072, out_features=5, bias=False)
  (layers): ModuleList(
    (0-17): 18 x Linear(in_features=5, out_features=5, bias=False)
  )
  (final): Linear(in_features=5, out_features=10, bias=False)
)

In [67]:
mlp

MLP(
  (initial): Linear(in_features=3072, out_features=2, bias=False)
  (layers): ModuleList(
    (0-2): 3 x Linear(in_features=2, out_features=2, bias=False)
  )
  (final): Linear(in_features=2, out_features=10, bias=False)
)

## Define the train and test loop

In [80]:
mlp = MLP(5, 4096)
print(mlp.to("mps"))

MLP(
  (initial): Linear(in_features=3072, out_features=4096, bias=False)
  (layers): ModuleList(
    (0-2): 3 x Linear(in_features=4096, out_features=4096, bias=False)
  )
  (final): Linear(in_features=4096, out_features=10, bias=False)
)


In [108]:
def loop_gemini(net, train, eta):
    dataloader = train_loader if train else test_loader
    description = "Training... " if train else "Testing... "

    acc_list = []

    for data, target in tqdm(dataloader, desc=description):
        data, target = data.to("mps"), target.to("mps")
        output = net(data)
        # print(data)

        loss = (
            output.logsumexp(dim=1).mean()
            - output[range(target.shape[0]), target].mean()
        )  # cross-entropy loss
        acc = (output.max(dim=1)[1] == target).sum() / target.shape[0]  # accuracy
        # print(acc.item())
        acc_list.append(acc.item())

        if train:
            loss.backward()

            for p in net.parameters():
                # print(p.shape)
                fan_out, fan_in = p.shape
                update = torch.sign(p.grad)
                # print(update)
                ## START BLOCK that you should modify
                update *= fan_out / fan_in
                ## END BLOCK that you should modify
                p.data -= eta * update
            net.zero_grad()
    # print(acc_list)
    return np.mean(acc_list)

## Sweep the learning rate at small width

Run the learning rate sweep at small width in the penultimate cell, then run the large
width training in the final cell using the best learning rate from the small width sweep.
Does the learning rate transfer well?
Write down the two blocks of PyTorch code that you modified in your solution
document, and report your finding.



In [103]:
def loop(net, train, eta):
    dataloader = train_loader if train else test_loader
    description = "Training... " if train else "Testing... "

    acc_list = []

    for data, target in tqdm(dataloader, desc=description):
        data, target = data.to("mps"), target.to("mps")
        output = net(data)

        loss = (
            output.logsumexp(dim=1).mean()
            - output[range(target.shape[0]), target].mean()
        )  # cross-entropy loss
        acc = (output.max(dim=1)[1] == target).sum() / target.shape[0]  # accuracy
        acc_list.append(acc.item())

        if train:
            loss.backward()

            for p in net.parameters():
                fan_out, fan_in = p.shape
                print(fan_out, fan_in)
                update = torch.sign(p.grad)
                ## START BLOCK that you should modify
                update *= math.sqrt(fan_out / fan_in) * torch.linalg.matrix_norm(update)
                ## END BLOCK that you should modify
                p.data -= eta * update
            net.zero_grad()

    return np.mean(acc_list)

In [102]:
def initialize_matrix_gemini(p):
    """
    Initializes a 2D weight matrix parameter following:
    W_k = (d_k / d_{k-1}) * M_k, where M_k is a random semi-orthogonal matrix.
    """
    if p.dim() < 2:
        # Safeguard: Fallback to standard uniform initialization for 1D biases
        torch.nn.init.uniform_(p.data, -0.1, 0.1)
        return

    # Extract dimensions: d_k (fan_out), d_{k-1} (fan_in)
    d_k, d_k1 = p.shape

    # 1. Sample a random semi-orthogonal matrix M_k directly on Apple Silicon GPU
    M_k = torch.empty(d_k, d_k1, device="mps")
    torch.nn.init.orthogonal_(M_k)

    # 2. Compute scale factor: (d_k / d_{k-1})
    scale_factor = d_k / d_k1

    # 3. Set the final weight matrix: W_k = scale_factor * M_k
    p.data.copy_(scale_factor * M_k)
    # print(p.shape,"\n")

In [99]:
def initialize_matrix(p):
    fan_out, fan_in = p.shape
    ## START BLOCK that you should modify ##
    p.data = math.sqrt(fan_out / fan_in) * torch.randn_like(
        torch.nn.init.orthogonal_(p)
    )
    # END BLOCK that you should modify

In [109]:
depth = 5
width = 32

import torch

accuracies = {"train_acc": [], "test_acc": [], "eta": []}
for eta in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    print(f"Training at {width=}, {depth=}, {eta=}")

    net = MLP(depth, width).to(device="mps")

    # print("\nNetwork tensor shapes are:\n")
    for name, p in net.named_parameters():
        # print(p.shape, "\t", name)
        initialize_matrix_gemini(p)

    train_acc = loop_gemini(net, train=True, eta=eta)
    test_acc = loop_gemini(net, train=False, eta=None)
    accuracies["train_acc"].append(train_acc)
    accuracies["test_acc"].append(test_acc)
    accuracies["eta"].append(eta)
    print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
    print("===================================================================\n")

Training at width=32, depth=5, eta=0.0001


Training... :   0%|          | 0/157 [00:00<?, ?it/s]

/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.181 and test acc=0.193


Training at width=32, depth=5, eta=0.001


...

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.114 and test acc=0.100


Training at width=32, depth=5, eta=1.0


Training... :   0%|          | 0/157 [00:00<?, ?it/s]

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.101 and test acc=0.100




In [110]:
depth = 5
width = 32

import torch

accuracies = {"train_acc": [], "test_acc": [], "eta": []}
for eta in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    print(f"Training at {width=}, {depth=}, {eta=}")

    net = MLP(depth, width).to(device="mps")

    # print("\nNetwork tensor shapes are:\n")
    for name, p in net.named_parameters():
        # print(p.shape, "\t", name)
        initialize_matrix(p)

    train_acc = loop(net, train=True, eta=eta)
    test_acc = loop(net, train=False, eta=None)
    accuracies["train_acc"].append(train_acc)
    accuracies["test_acc"].append(test_acc)
    accuracies["eta"].append(eta)
    print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
    print("===================================================================\n")

Training at width=32, depth=5, eta=0.0001


Training... :   0%|          | 0/157 [00:00<?, ?it/s]

/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.121 and test acc=0.112


Training at width=32, depth=5, eta=0.001


...

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.100 and test acc=0.100


Training at width=32, depth=5, eta=1.0


Training... :   0%|          | 0/157 [00:00<?, ?it/s]

Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.098 and test acc=0.100




In [105]:
import pandas as pd

gemini2 = pd.DataFrame.from_dict(accuracies)
gemini2

,train_acc,test_acc,eta
0,0.288067,0.338311,0.0001
1,0.101662,0.100277,0.0010
2,0.098726,0.100277,0.0100
3,0.100518,0.100277,0.1000
4,0.099423,0.100475,1.0000


In [111]:
import pandas as pd

df = pd.DataFrame.from_dict(accuracies)
df

,train_acc,test_acc,eta
0,0.121218,0.111551,0.0001
1,0.101214,0.100376,0.0010
2,0.100070,0.100277,0.0100
3,0.099871,0.100277,0.1000
4,0.097731,0.100277,1.0000


## Run the best learning rate at large width

In [78]:
import pandas as pd

df = pd.DataFrame.from_dict(accuracies)

In [79]:
df

,train_acc,test_acc,eta
0,0.099572,0.100277,0.0001
1,0.095641,0.095036,0.0010
2,0.100169,0.094640,0.0090
3,0.099423,0.100277,0.0000
4,0.102906,0.096816,0.0100


## at eta .01 we get the best train and test accuracies

In [112]:
depth = 5
width = 4096

# START BLOCK you should set this to the best value of eta from the previous cell
eta = 0.0001
# END BLOCK

print(f"Training at {width=}, {depth=}, {eta=}")

net = MLP(depth, width).to("mps")

print("\nNetwork tensor shapes are:\n")
for name, p in net.named_parameters():
    print(p.shape, "\t", name)
    initialize_matrix(p)

train_acc = loop(net, train=True, eta=eta)
test_acc = loop(net, train=False, eta=None)

print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
print("===================================================================\n")

Training at width=4096, depth=5, eta=0.0001

Network tensor shapes are:

torch.Size([4096, 3072]) 	 initial.weight
torch.Size([4096, 4096]) 	 layers.0.weight
torch.Size([4096, 4096]) 	 layers.1.weight
torch.Size([4096, 4096]) 	 layers.2.weight
torch.Size([10, 4096]) 	 final.weight


Training... :   0%|          | 0/157 [00:00<?, ?it/s]

/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Testing... :   0%|          | 0/79 [00:00<?, ?it/s]


We achieved train acc=0.147 and test acc=0.102


